In [1]:
pip install openai anthropic google-generativeai openpyxl pandas

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 8.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 19.1 MB/s eta 0:00:00
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 31.1 MB/s eta 0:00:00 0:00:01
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.32.1
    Uninstalling proto

In [ ]:
import os

os.environ["ANTHROPIC_API_KEY"] = "MY_KEY"
os.environ["OPENAI_API_KEY"] = "MY_KEY"
os.environ["GOOGLE_API_KEY"] = "MY_KEY"
os.environ["DEEPSEEK_API_KEY"] = "MY_KEY"

In [ ]:
import os
import re
import time
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side
 
# ── Configuration ─────────────────────────────────────────────────────────────
 
DATASET_FILE = "dataset_gender_switching.xlsx"
OUTPUT_FILE  = "results_gender_switching.xlsx"
 
MODELS = [
    "gpt-4o",
    # "claude-sonnet-4-6",
    # "gemini-2.0-flash",
    # "deepseek-chat",
]
 
RUNS_PER_HYPOTHESIS = {
    "H0": 10,
    "H2": 10,
    "H3": 10,
    "H4": 5,
    "H5": 10,
    "H6": 5,
}
DEFAULT_RUNS = 5
 
TEMPERATURE = {
    "gpt-4o":             1.2,
    "claude-sonnet-4-6":  1.0,
    "gemini-2.0-flash":   1.0,
    "deepseek-chat":      1.2,
}
DEFAULT_TEMPERATURE = 1.0
 
SLEEP_BETWEEN_CALLS = 1.5
 
#SYSTEM_PROMPT = "Отвечай на русском языке."
 
# Regex for illegal Excel characters
ILLEGAL_CHARS_RE = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f]')
 
 
# ── Text cleaning ─────────────────────────────────────────────────────────────
 
def clean(value):
    """Remove characters that openpyxl cannot write to Excel."""
    if isinstance(value, str):
        return ILLEGAL_CHARS_RE.sub('', value)
    return value
 
 
# ── Model clients ─────────────────────────────────────────────────────────────
 
def call_openai(messages, model, temperature):
    from openai import OpenAI
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    full_messages = [{"role": "system", "content": SYSTEM_PROMPT}] + messages
    response = client.chat.completions.create(
        model=model,
        messages=full_messages,
        max_tokens=512,
        temperature=temperature,
    )
    return response.choices[0].message.content
 
 
def call_anthropic(messages, model, temperature):
    import anthropic
    client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    response = client.messages.create(
        model=model,
        max_tokens=512,
        system=SYSTEM_PROMPT,
        messages=messages,
        temperature=min(temperature, 1.0),
    )
    return response.content[0].text
 
 
def call_gemini(messages, model, temperature):
    import google.generativeai as genai
    genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
    gemini_model = genai.GenerativeModel(
        model_name=model,
        system_instruction=SYSTEM_PROMPT,
        generation_config={"temperature": temperature, "max_output_tokens": 512},
    )
    history = []
    last_user_msg = ""
    for msg in messages:
        if msg["role"] == "user":
            last_user_msg = msg["content"]
        elif msg["role"] == "assistant":
            history.append({"role": "user",  "parts": [last_user_msg]})
            history.append({"role": "model", "parts": [msg["content"]]})
            last_user_msg = ""
    chat = gemini_model.start_chat(history=history)
    return chat.send_message(last_user_msg).text
 
 
def call_deepseek(messages, model, temperature):
    from openai import OpenAI
    client = OpenAI(
        api_key=os.environ["DEEPSEEK_API_KEY"],
        base_url="https://api.deepseek.com",
    )
    full_messages = [{"role": "system", "content": SYSTEM_PROMPT}] + messages
    response = client.chat.completions.create(
        model=model,
        messages=full_messages,
        max_tokens=512,
        temperature=temperature,
    )
    return response.choices[0].message.content
 
 
def call_model(messages, model):
    temperature = TEMPERATURE.get(model, DEFAULT_TEMPERATURE)
    if model.startswith("claude"):
        return call_anthropic(messages, model, temperature)
    elif model.startswith("gemini"):
        return call_gemini(messages, model, temperature)
    elif model.startswith("deepseek"):
        return call_deepseek(messages, model, temperature)
    else:
        return call_openai(messages, model, temperature)
 
 
# ── Dialogue runners ──────────────────────────────────────────────────────────
 
def run_single_turn(turn1, model):
    return call_model([{"role": "user", "content": turn1}], model)
 
 
def run_two_turn(turn1, turn2, model):
    messages = [{"role": "user", "content": turn1}]
    resp1 = call_model(messages, model)
    messages += [
        {"role": "assistant", "content": resp1},
        {"role": "user",      "content": turn2},
    ]
    resp2 = call_model(messages, model)
    return resp1, resp2
 
 
def run_three_turn(turn1, turn2, turn3, model):
    messages = [{"role": "user", "content": turn1}]
    resp1 = call_model(messages, model)
    messages += [{"role": "assistant", "content": resp1},
                 {"role": "user",      "content": turn2}]
    resp2 = call_model(messages, model)
    messages += [{"role": "assistant", "content": resp2},
                 {"role": "user",      "content": turn3}]
    resp3 = call_model(messages, model)
    return resp1, resp2, resp3
 
 
# ── Dataset loading ───────────────────────────────────────────────────────────
 
def load_dataset(path):
    df = pd.read_excel(path, sheet_name="Dataset")
    df.columns = [c.strip() for c in df.columns]
    for col in ["Turn_1_User", "Turn_2_User", "Turn_3_User"]:
        if col in df.columns:
            df[col] = df[col].fillna("")
    return df
 
 
# ── Results saving ────────────────────────────────────────────────────────────
 
def save_results(results, path):
    df = pd.DataFrame(results)
 
    # Clean all string columns — remove characters Excel cannot handle
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].apply(clean)
 
    cols = [
        "prompt_id", "model", "hypothesis", "topic", "topic_category",
        "tone", "gender_signal", "trigger_turn", "run_number", "condition",
        "response_t1", "response_t2", "response_t3",
        "response_gender_t1", "response_gender_t2", "response_gender_t3",
        "gender_shift", "marker_word_t1", "marker_word_t2", "marker_word_t3",
        "note",
    ]
    for c in cols:
        if c not in df.columns:
            df[c] = ""
    df = df[cols]
 
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name="Results", index=False)
        df_ann = df[["prompt_id", "model", "condition", "run_number",
                     "response_t1", "response_t2", "response_t3"]].copy()
        for col in ["response_gender_t1", "response_gender_t2", "response_gender_t3",
                    "gender_shift", "marker_word_t1", "marker_word_t2", "note"]:
            df_ann[col] = ""
        df_ann.to_excel(writer, sheet_name="Annotation", index=False)
 
    wb = load_workbook(path)
    thin = Side(style="thin", color="000000")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    for sheet_name in ["Results", "Annotation"]:
        ws = wb[sheet_name]
        for cell in ws[1]:
            cell.font = Font(name="Arial", bold=True, size=11)
            cell.alignment = Alignment(horizontal="center", wrap_text=True)
            cell.border = border
        ws.freeze_panes = "A2"
    wb.save(path)
    print(f"  → Saved to {path}")
 
 
# ── Main ──────────────────────────────────────────────────────────────────────
 
def main():
    print(f"Loading dataset from {DATASET_FILE}...")
    df = load_dataset(DATASET_FILE)
    print(f"  {len(df)} prompts loaded")
 
    total = sum(
        RUNS_PER_HYPOTHESIS.get(row["Hypothesis"], DEFAULT_RUNS) * len(MODELS) * 2
        for _, row in df.iterrows()
    )
    print(f"  Estimated total API calls: {total}")
    print(f"  Estimated time: ~{total * (SLEEP_BETWEEN_CALLS + 3) / 60:.0f} min\n")
 
    results = []
    done = 0
 
    for _, row in df.iterrows():
        pid    = row["ID"]
        hyp    = row.get("Hypothesis", "")
        turn1  = row["Turn_1_User"]
        turn2  = row.get("Turn_2_User", "")
        turn3  = row.get("Turn_3_User", "")
        has_t3 = bool(turn3)
        n_runs = RUNS_PER_HYPOTHESIS.get(hyp, DEFAULT_RUNS)
 
        base = {
            "prompt_id":      pid,
            "hypothesis":     hyp,
            "topic":          row.get("Topic", ""),
            "topic_category": row.get("Topic_Category", ""),
            "tone":           row.get("Tone", ""),
            "gender_signal":  row.get("Gender_Signal", ""),
            "trigger_turn":   row.get("Trigger_Turn", ""),
            "response_gender_t1": "", "response_gender_t2": "",
            "response_gender_t3": "", "gender_shift": "",
            "marker_word_t1": "", "marker_word_t2": "",
            "marker_word_t3": "", "note": "",
        }
 
        for model in MODELS:
            for run in range(1, n_runs + 1):
 
                # ── Single-turn ───────────────────────────────────────────────
                try:
                    resp1 = run_single_turn(turn1, model)
                    results.append({**base,
                        "model": model, "run_number": run,
                        "condition": "single-turn",
                        "response_t1": resp1, "response_t2": "", "response_t3": "",
                    })
                    done += 1
                    print(f"[{done}/{total}] {pid} | {model} | run {run}/{n_runs} | single-turn ✓")
                except Exception as e:
                    print(f"  ERROR {pid} {model} single-turn: {e}")
                    results.append({**base,
                        "model": model, "run_number": run,
                        "condition": "single-turn",
                        "response_t1": f"ERROR: {e}", "response_t2": "", "response_t3": "",
                    })
                time.sleep(SLEEP_BETWEEN_CALLS)
 
                # ── Two-turn / three-turn ─────────────────────────────────────
                try:
                    if has_t3:
                        r1, r2, r3 = run_three_turn(turn1, turn2, turn3, model)
                        cond = "three-turn"
                    else:
                        r1, r2 = run_two_turn(turn1, turn2, model)
                        r3 = ""
                        cond = "two-turn"
                    results.append({**base,
                        "model": model, "run_number": run,
                        "condition": cond,
                        "response_t1": r1, "response_t2": r2, "response_t3": r3,
                    })
                    done += 1
                    print(f"[{done}/{total}] {pid} | {model} | run {run}/{n_runs} | {cond} ✓")
                except Exception as e:
                    print(f"  ERROR {pid} {model} two-turn: {e}")
                    results.append({**base,
                        "model": model, "run_number": run,
                        "condition": "two-turn",
                        "response_t1": f"ERROR: {e}", "response_t2": "", "response_t3": "",
                    })
                time.sleep(SLEEP_BETWEEN_CALLS)
 
        # Save every 20 completed calls to avoid losing data on crash
        if done % 20 == 0 and done > 0:
            save_results(results, OUTPUT_FILE)
 
    save_results(results, OUTPUT_FILE)
    print(f"\nDone. {len(results)} rows total.")

In [16]:
if __name__ == "__main__":
    main()

Loading dataset from dataset_gender_switching.xlsx...
  81 prompts loaded
  Estimated total API calls: 1420
  Estimated time: ~106 min

[1/1420] H0_1 | gpt-4o | run 1/10 | single-turn ✓
[2/1420] H0_1 | gpt-4o | run 1/10 | two-turn ✓
[3/1420] H0_1 | gpt-4o | run 2/10 | single-turn ✓
[4/1420] H0_1 | gpt-4o | run 2/10 | two-turn ✓
[5/1420] H0_1 | gpt-4o | run 3/10 | single-turn ✓
[6/1420] H0_1 | gpt-4o | run 3/10 | two-turn ✓
[7/1420] H0_1 | gpt-4o | run 4/10 | single-turn ✓
[8/1420] H0_1 | gpt-4o | run 4/10 | two-turn ✓
[9/1420] H0_1 | gpt-4o | run 5/10 | single-turn ✓
[10/1420] H0_1 | gpt-4o | run 5/10 | two-turn ✓
[11/1420] H0_1 | gpt-4o | run 6/10 | single-turn ✓
[12/1420] H0_1 | gpt-4o | run 6/10 | two-turn ✓
[13/1420] H0_1 | gpt-4o | run 7/10 | single-turn ✓
[14/1420] H0_1 | gpt-4o | run 7/10 | two-turn ✓
[15/1420] H0_1 | gpt-4o | run 8/10 | single-turn ✓
[16/1420] H0_1 | gpt-4o | run 8/10 | two-turn ✓
[17/1420] H0_1 | gpt-4o | run 9/10 | single-turn ✓
[18/1420] H0_1 | gpt-4o | run 